# 08. UGV IMU 및 위치 확인 (IMU)

영상 대응: **28:14–30:00**

> 3차 교육 영상 20–30분의 기능을 재구성한 **로컬 실습본**입니다. 공식 원본 노트북이 아닙니다. `mil_msgs` 원본 정의가 없어 이 환경에서는 `sim_mil_interfaces`를 사용합니다. 실제 ROS 2로 시뮬레이터와 통신합니다.

`/imu/data`의 `sensor_msgs/msg/Imu`와 `/odom`의 `nav_msgs/msg/Odometry`를 구독합니다. 영상 기준 갱신 주기는 20Hz입니다.

| 값 | 의미 |
|---|---|
| roll | 좌우 기울기(°) |
| pitch | 앞뒤 기울기(°) |
| yaw | 로봇 회전 방향(°) |
| linear_acceleration | 가속도(m/s²) |
| g | 중력가속도 기준 크기 |
| position.x / y | 출발점 기준 전후 / 좌우 위치(m) |

영상 예제는 센서를 읽은 뒤 0속도 명령도 보냅니다. 현재 시뮬레이터는 yaw 기반의 이상적인 자세를 제공하며 가속도·각속도는 제공하지 않습니다(`covariance[0] == -1`). 따라서 가속도를 0g 또는 1g로 꾸며 표시하지 않습니다.

서비스 응답 대기에는 시간 제한을 두고, 센서가 없거나 오래된 경우를 구분해야 한다는 영상 설명을 실습 코드에 적용했습니다.


## 실행 방법

위쪽 커널을 **Python 3 (ROS 2 · simulation)** 으로 선택하고, 아래 셀을 위에서부터 **Shift+Enter**로 실행합니다. 물리 경기장과 ROS 실습 연결을 먼저 켜세요. 각 실습은 시간이 지나면 종료되며, 셀을 다시 실행할 수 있습니다.


In [ ]:
import time
import math
import uuid
import rclpy
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, qos_profile_sensor_data
from rclpy.signals import SignalHandlerOptions
from sim_mil_interfaces.msg import State as MissionState, Match as MatchCommand

# 상태/경기 토픽은 마지막 값을 새 구독자에게도 전달합니다.
STATUS_QOS = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                        durability=DurabilityPolicy.TRANSIENT_LOCAL)

def new_node(prefix):
    if not rclpy.ok():
        # Jupyter의 중단 버튼이 KeyboardInterrupt와 finally 정지를 처리하게 둡니다.
        rclpy.init(args=[], signal_handler_options=SignalHandlerOptions.NO)
    return rclpy.create_node(prefix + '_' + uuid.uuid4().hex[:8])

def spin_until(node, predicate, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not predicate() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=min(0.05, max(0.0, deadline-time.monotonic())))
    if not predicate():
        raise TimeoutError('ROS 데이터/응답 시간 초과. 물리 경기장과 ROS 연결 상태를 확인하세요.')

def watch_status(node):
    data = {'state': None, 'match': None, 'state_at': None}
    def on_state(msg):
        data['state'] = msg
        data['state_at'] = time.monotonic()
    def on_match(msg):
        data['match'] = msg
    subscriptions = [
        node.create_subscription(MissionState, '/mil/state', on_state, STATUS_QOS),
        node.create_subscription(MatchCommand, '/mil/match', on_match, STATUS_QOS),
    ]
    spin_until(node, lambda: data['state'] is not None and data['match'] is not None)
    return data, subscriptions

def allowed(data):
    if data['state_at'] is None or time.monotonic()-data['state_at'] > 1.5:
        return False
    match = data['match']
    if match is None or match.cmd == 'pause':
        return False
    return match.mode == 'practice' or (match.mode == 'mission' and match.cmd in ('start', 'resume'))

def require_control(data):
    if not allowed(data):
        raise RuntimeError('제어 잠금 또는 상태 데이터 지연. 경기 재개·연결 상태를 확인하세요.')

def wait_publisher(node, publisher, timeout=5.0):
    spin_until(node, lambda: publisher.get_subscription_count() > 0, timeout)

def yaw_degrees(q):
    return math.degrees(math.atan2(2.0*(q.w*q.z + q.x*q.y), 1.0-2.0*(q.y*q.y+q.z*q.z)))

def source_age_seconds(node, message, maximum, label):
    stamp = message.header.stamp
    source_ns = int(stamp.sec)*1_000_000_000 + int(stamp.nanosec)
    if source_ns <= 0:
        raise ValueError(label + ': 원본 센서 시각이 없습니다.')
    age = (node.get_clock().now().nanoseconds-source_ns)/1_000_000_000
    if age < -0.25:
        raise ValueError(f'{label}: 센서 시각이 {-age:.3f}초 미래입니다. ROS 시계 설정을 확인하세요.')
    if age > maximum:
        raise TimeoutError(f'{label}: 원본 데이터가 {age:.3f}초 전 값입니다.')
    return age


## 실습 설정

아래 값을 바꾸고 다시 실행할 수 있습니다.


In [ ]:
TIMEOUT_SECONDS = 5.0
MAX_SENSOR_AGE_SECONDS = 0.75
SEND_STOP = True


## 실행


In [ ]:
from sensor_msgs.msg import Imu
from nav_msgs.msg import Odometry
from geometry_msgs.msg import Twist
node = new_node('demo_imu')
latest = {'imu':None,'odom':None,'imu_at':None,'odom_at':None}
publisher = None
def on_imu(msg):
    latest['imu'], latest['imu_at'] = msg, time.monotonic()
def on_odom(msg):
    latest['odom'], latest['odom_at'] = msg, time.monotonic()
try:
    subscriptions = [node.create_subscription(Imu, '/imu/data', on_imu, qos_profile_sensor_data),
                     node.create_subscription(Odometry, '/odom', on_odom, qos_profile_sensor_data)]
    spin_until(node, lambda: latest['imu'] is not None and latest['odom'] is not None, TIMEOUT_SECONDS)
    now = time.monotonic()
    if any(now-latest[key] > MAX_SENSOR_AGE_SECONDS for key in ('imu_at','odom_at')):
        raise TimeoutError('IMU 또는 위치 데이터 지연')
    source_ages = {key:source_age_seconds(node, latest[key], MAX_SENSOR_AGE_SECONDS, key) for key in ('imu','odom')}
    imu = latest['imu'];q = imu.orientation
    if imu.orientation_covariance[0] == -1:
        raise RuntimeError('자세 데이터를 제공하지 않는 IMU 메시지입니다.')
    roll = math.degrees(math.atan2(2*(q.w*q.x+q.y*q.z), 1-2*(q.x*q.x+q.y*q.y)))
    pitch = math.degrees(math.asin(max(-1.0,min(1.0,2*(q.w*q.y-q.z*q.x)))))
    yaw = yaw_degrees(q)
    position = latest['odom'].pose.pose.position
    acceleration_available = bool(imu.linear_acceleration_covariance[0] != -1)
    acceleration_g = math.sqrt(imu.linear_acceleration.x**2+imu.linear_acceleration.y**2+imu.linear_acceleration.z**2)/9.80665 if acceleration_available else None
    print(f'Roll: {roll:.1f} | Pitch: {pitch:.1f} | Yaw: {yaw:.1f} | X: {position.x:.3f} | Y: {position.y:.3f}')
    print(f'Acc: {acceleration_g:.2f}g' if acceleration_available else 'Acc: 미제공 (시뮬레이션 IMU)')
    if SEND_STOP:
        publisher = node.create_publisher(Twist, '/cmd_vel', 10)
        wait_publisher(node, publisher)
        publisher.publish(Twist())
        time.sleep(0.1)
        print('Stop command sent')
    DEMO_RESULT = {'demo':'08','roll_deg':roll,'pitch_deg':pitch,'yaw_deg':yaw,
                   'xy':[float(position.x),float(position.y)],'acceleration_available':acceleration_available,
                   'acceleration_g':acceleration_g,'stop_sent':bool(SEND_STOP),
                   'source_age_seconds':source_ages,'received_age_seconds':{key:now-latest[key+'_at'] for key in ('imu','odom')}}
finally:
    if publisher is not None:
        publisher.publish(Twist())
        time.sleep(0.03)
    node.destroy_node()


[← 실습 목차](00.실습목차.ipynb) · [코드 편집](../team_node.py)

실물 장치의 토픽 타입·주기·응답과 맞는지는 지급 UGV에서 확인해야 합니다.
